In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 0 — Montar Drive y obtener modelo_condominio_final.pt
# Antes de correr: sube modelo_condominio_final.pt a tu Google Drive
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Ajusta el nombre si lo guardaste con otro nombre
RUTA_MODELO_BASE = '/content/drive/MyDrive/modelo_condominio_final.pt'

if os.path.exists(RUTA_MODELO_BASE):
    shutil.copy(RUTA_MODELO_BASE, '/content/modelo_condominio_final.pt')
    print('✅ modelo_condominio_final.pt copiado a /content/')
else:
    print('❌ No se encontró el modelo base. Verifica la ruta en tu Drive.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 1 — Configurar Kaggle usando Colab Secrets
#
# Antes de correr:
#   1. Clic en el ícono de llave (🔑) en el panel izquierdo de Colab
#   2. Agregar secret: Name = KAGGLE_TOKEN, Value = tu token KGAT_...
#   3. Activar el toggle "Notebook access"
# ─────────────────────────────────────────────────────────────────────────────
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = 'roly'
os.environ['KAGGLE_TOKEN']    = userdata.get('KAGGLE_TOKEN')

!pip install kaggle -q
print('✅ Kaggle configurado')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 2 — Descargar datasets
#
# Dataset 1: Road Issues Detection (9.660 imgs de problemas viales incluyendo
#            estacionamiento ilegal)  →  clase: infraccion
#
# Dataset 2: PKLot (12.416 imgs de parqueos de cámara de seguridad con
#            vehículos correctamente estacionados)  →  clase: normal
# ─────────────────────────────────────────────────────────────────────────────
print('📥 Descargando Road Issues Detection Dataset...')
!kaggle datasets download -d programmerrdai/road-issues-detection-dataset -p /content/road_issues --unzip -q
print('✅ Road Issues descargado')

print('\n📥 Descargando PKLot Dataset...')
!kaggle datasets download -d ammarnassanalhajali/pklot-dataset -p /content/pklot --unzip -q
print('✅ PKLot descargado')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 3 — Explorar estructura descargada
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path

print('=== ROAD ISSUES ===')
road_root = Path('/content/road_issues')
for item in sorted(road_root.rglob('*')):
    if len(item.relative_to(road_root).parts) <= 2:
        count = len(list(item.iterdir())) if item.is_dir() else ''
        nivel = len(item.relative_to(road_root).parts)
        print('  ' * (nivel - 1) + item.name + (f'/  [{count} items]' if item.is_dir() else ''))

print('\n=== PKLot ===')
pklot_root = Path('/content/pklot')
for item in sorted(pklot_root.rglob('*')):
    if len(item.relative_to(pklot_root).parts) <= 3:
        count = len(list(item.iterdir())) if item.is_dir() else ''
        nivel = len(item.relative_to(pklot_root).parts)
        print('  ' * (nivel - 1) + item.name + (f'/  [{count} items]' if item.is_dir() else ''))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 4 — Organizar imágenes en train/val por clase
#
# Estructura final:
#   /content/dataset/train/infraccion/   ← imgs de problemas viales
#   /content/dataset/train/normal/       ← imgs de parqueo correcto
#   /content/dataset/val/infraccion/
#   /content/dataset/val/normal/
#
# Límite por clase: 3000 imgs máximo para que el entrenamiento sea rápido
# en Colab (GPU T4 gratis tiene límite de tiempo).
# ─────────────────────────────────────────────────────────────────────────────
import random
import shutil
from pathlib import Path

random.seed(42)
LIMITE_POR_CLASE = 3000
SPLIT_VAL        = 0.20   # 20% validación
EXTENSIONES      = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

def organizar_clase(origen_dirs, clase, limite=LIMITE_POR_CLASE):
    """Copia imágenes de origen_dirs a train/val/<clase>/"""
    imagenes = []
    for d in origen_dirs:
        imagenes += [p for p in Path(d).rglob('*') if p.suffix in EXTENSIONES]

    random.shuffle(imagenes)
    imagenes = imagenes[:limite]

    split_idx = int(len(imagenes) * (1 - SPLIT_VAL))
    splits = [('train', imagenes[:split_idx]), ('val', imagenes[split_idx:])]

    for split_name, imgs in splits:
        dst = Path(f'/content/dataset/{split_name}/{clase}')
        dst.mkdir(parents=True, exist_ok=True)
        for i, img in enumerate(imgs):
            shutil.copy(img, dst / f'{clase}_{i:05d}{img.suffix.lower()}')
        print(f'  {split_name}/{clase}: {len(imgs)} imágenes')

# ── Clase: infraccion ────────────────────────────────────────────────────────
# Usamos TODAS las imágenes del Road Issues Dataset (problemas viales = infracciones)
print('Organizando clase: infraccion...')
organizar_clase(
    origen_dirs=['/content/road_issues'],
    clase='infraccion'
)

# ── Clase: normal ────────────────────────────────────────────────────────────
# Usamos las imágenes del PKLot (parqueos de cámara de seguridad = parqueo correcto)
print('\nOrganizando clase: normal...')
organizar_clase(
    origen_dirs=['/content/pklot'],
    clase='normal'
)

# Resumen
print('\n=== RESUMEN DATASET ===')
for split in ['train', 'val']:
    for clase in ['infraccion', 'normal']:
        n = len(list(Path(f'/content/dataset/{split}/{clase}').glob('*')))
        print(f'  {split}/{clase}: {n} imágenes')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 5 — Preparar dataloaders
# ─────────────────────────────────────────────────────────────────────────────
import torch
from torchvision import datasets, transforms

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder('/content/dataset/train', transform=transform_train)
val_ds   = datasets.ImageFolder('/content/dataset/val',   transform=transform_val)

# ImageFolder ordena alfabéticamente: infraccion=0, normal=1
print(f'Clases detectadas: {train_ds.classes}')  # ['infraccion', 'normal']
print(f'Imágenes entrenamiento: {len(train_ds)}')
print(f'Imágenes validación:    {len(val_ds)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 6 — Cargar modelo_condominio_final.pt como base y entrenar
#
# Pipeline:
#   ImageNet (PyTorch)
#       ↓ fine-tune → modelo_condominio_final.pt  (ya sabe: mascotas/personas/vacios/vehiculos)
#           ↓ fine-tune → modelo_vehiculo_estacionamiento.pth  (infraccion/normal)
#
# Ventaja: el modelo ya conoce cómo lucen los vehículos en el condominio.
# Converge más rápido que partir desde ImageNet.
# ─────────────────────────────────────────────────────────────────────────────
import time
import torch.nn as nn
from torchvision import models

# 1. Reconstruir arquitectura del modelo base (4 clases condominio)
modelo = models.resnet18(weights=None)
modelo.fc = nn.Linear(modelo.fc.in_features, 4)   # arquitectura original del condominio

# 2. Cargar pesos del modelo condominio
modelo.load_state_dict(torch.load('/content/modelo_condominio_final.pt', map_location=device))
print('✅ modelo_condominio_final.pt cargado')

# 3. Reemplazar capa final: 4 clases → 2 clases (infraccion / normal)
modelo.fc = nn.Linear(modelo.fc.in_features, 2)
modelo = modelo.to(device)

# 4. Configurar entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print('\n🚀 Iniciando entrenamiento...')
print('-' * 45)

start = time.time()
for epoch in range(10):
    # ── Entrenamiento ────────────────────────────────
    modelo.train()
    loss_total = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(modelo(imgs), labels)
        loss.backward()
        optimizer.step()
        loss_total += loss.item()

    # ── Validación rápida ────────────────────────────
    modelo.eval()
    correctos = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = modelo(imgs).argmax(dim=1)
            correctos += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correctos / total * 100
    scheduler.step()
    print(f'Época {epoch+1:02d}/10 — Loss: {loss_total/len(train_loader):.4f}  |  Val Acc: {acc:.1f}%')

print('-' * 45)
print(f'🏆 Entrenamiento terminado en {(time.time()-start)/60:.1f} minutos')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 7 — Evaluación detallada por clase
# ─────────────────────────────────────────────────────────────────────────────
from collections import defaultdict

modelo.eval()
clases = train_ds.classes  # ['infraccion', 'normal']
correctos_clase = defaultdict(int)
total_clase     = defaultdict(int)

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = modelo(imgs).argmax(dim=1)
        for pred, label in zip(preds, labels):
            total_clase[label.item()]     += 1
            correctos_clase[label.item()] += int(pred == label)

print('=== PRECISIÓN POR CLASE ===')
for idx, nombre in enumerate(clases):
    if total_clase[idx] > 0:
        acc = correctos_clase[idx] / total_clase[idx] * 100
        print(f'  {nombre:12s}: {acc:.1f}%  ({correctos_clase[idx]}/{total_clase[idx]} correctas)')

total_global = sum(total_clase.values())
aciertos     = sum(correctos_clase.values())
print(f'\n  TOTAL        : {aciertos/total_global*100:.1f}%  ({aciertos}/{total_global})')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 8 — Guardar modelo + subir a Google Drive
# ─────────────────────────────────────────────────────────────────────────────
torch.save(modelo.state_dict(), 'modelo_vehiculo_estacionamiento.pth')
print("✅ modelo_vehiculo_estacionamiento.pth guardado")

shutil.copy('modelo_vehiculo_estacionamiento.pth',
            '/content/drive/MyDrive/modelo_vehiculo_estacionamiento.pth')
print("✅ Respaldado en Google Drive")

# Opcional: descargar directo al computador
# from google.colab import files
# files.download('modelo_vehiculo_estacionamiento.pth')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 9 — Prueba rápida con una imagen
# Sube una imagen de prueba a Colab (vehículo mal estacionado o bien estacionado)
# y ejecuta esta celda para ver si el modelo clasifica correctamente.
# ─────────────────────────────────────────────────────────────────────────────
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

def predecir(ruta_imagen):
    modelo.eval()
    img = Image.open(ruta_imagen).convert('RGB')
    tensor = transform_val(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = modelo(tensor)
        probs  = F.softmax(logits, dim=1)[0]

    p_infraccion = probs[0].item() * 100
    p_normal     = probs[1].item() * 100
    clase_final  = clases[int(torch.argmax(probs).item())]

    print('-' * 35)
    print(f'🚗 Normal:     {p_normal:.1f}%')
    print(f'🚨 Infraccion: {p_infraccion:.1f}%')
    print('-' * 35)
    print(f'PREDICCIÓN: {clase_final.upper()}')

    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    color = 'red' if clase_final == 'infraccion' else 'green'
    plt.title(f'{clase_final.upper()} ({max(p_infraccion, p_normal):.1f}%)',
              fontsize=14, color=color)
    plt.axis('off')
    plt.show()

# Cambia 'test.jpg' por la imagen que subiste
predecir('test.jpg')